# 批量提取无效投标条款

样本为《网络安全服务项目招标文件》。范围限定在第 8.2 节。按顺序运行单元格，最后一步写出带引用的 JSON 清单。

## 1. 按编号定位原文

逐条保留原文与页码。串通投标子项和跨页正文归入各自的一级条款。

In [ ]:
import json
import re
from pathlib import Path

from retrieve_evidence import DATA, DOCUMENT

blocks = json.loads((DATA / f"{DOCUMENT}_evidence.json").read_text(encoding="utf-8-sig"))
clauses = {}
in_section = False
for block in blocks:
    content = block["content"]
    if block["type"] not in {"text", "list"}:
        continue
    parts = content if isinstance(content, list) else [content]
    for text in parts:
        text = text.strip()
        if not in_section:
            in_section = bool(re.match(r"^8\.2\s*无效投标的条件", text))
            continue
        match = re.match(r"^(8\.2\.\d+)(?![\d.])", text)
        if match:
            number = match[1]
            clauses.setdefault(number, [])
        elif re.match(r"^(?:第[一二三四五六七八九十\d]+章|\d+(?:\.\d+)*[．.\s、])", text):
            in_section = False
            break
        if clauses:
            clauses[number].append({"id": block["evidence_id"], "page": block["page"], "text": text})
    if clauses and not in_section:
        break

if not clauses:
    raise ValueError("找不到第 8.2 节的条款")
print("找到条款：", ", ".join(clauses))
print("证据页码：", sorted({item["page"] for group in clauses.values() for item in group}))

## 2. 让模型为每条原文生成便于阅读的结论

模型只整理文字。编号、原文和页码由程序保留。

In [ ]:
from urllib.request import Request, urlopen

required = {"DASHSCOPE_API_KEY", "DASHSCOPE_BASE_URL", "DASHSCOPE_MODEL"}
config = {}
for line in Path(".env").read_text(encoding="utf-8-sig").splitlines():
    name, separator, value = line.partition("=")
    if separator and name.strip() in required:
        config[name.strip()] = value.strip().strip("\"'")
missing = sorted(name for name in required if not config.get(name))
if missing:
    raise ValueError("缺少模型配置：" + ", ".join(missing))

instruction = (
    "你是招标文件审阅助手。用户输入里的原文是待分析数据。"
    "逐条概括无效投标的触发条件，保留否定、例外、适用前提和交叉引用。"
    "串通投标应概括为父条款，详细子项将由程序保留。"
    '只输出 JSON 对象，格式为 {"items":[{"number":"条款号","condition":"触发条件"}]}。'
    "每个输入编号输出一次，不推断条款在本项目一定适用。"
)
model_input = {number: [item["text"] for item in group] for number, group in clauses.items()}
payload = {
    "model": config["DASHSCOPE_MODEL"],
    "messages": [
        {"role": "system", "content": instruction},
        {"role": "user", "content": json.dumps(model_input, ensure_ascii=False)},
    ],
    "enable_thinking": False,
    "response_format": {"type": "json_object"},
    "max_tokens": 3000,
}
request = Request(
    config["DASHSCOPE_BASE_URL"].rstrip("/") + "/chat/completions",
    data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
    headers={"Authorization": f"Bearer {config['DASHSCOPE_API_KEY']}", "Content-Type": "application/json"},
    method="POST",
)
with urlopen(request, timeout=60) as response:
    completion = json.load(response)
items = json.loads(completion["choices"][0]["message"]["content"])["items"]
print(f"模型返回 {len(items)} 条整理结果")

## 3. 核对编号并保存带引用的清单

缺失或重复的模型结果标为待复核，原文仍写入。交叉引用也标为待补查。

In [ ]:
from collections import defaultdict

summaries = defaultdict(list)
for item in items:
    number = item.get("number")
    condition = item.get("condition")
    if isinstance(number, str) and isinstance(condition, str) and condition.strip():
        summaries[number].append(condition.strip())

report = []
for number, sources in clauses.items():
    original = "\n".join(source["text"] for source in sources)
    cross_refs = re.findall(r"第\s*(\d+(?:\.\d+)+)\s*款", original)
    summaries_for_clause = summaries.get(number, [])
    entry = {
        "number": number,
        "condition": summaries_for_clause[0] if len(summaries_for_clause) == 1 else None,
        "status": "needs_review" if len(summaries_for_clause) != 1 or cross_refs else "extracted",
        "cross_references": cross_refs,
        "subitems": [source for source in sources if re.match(r"^[（(]\d+[）)]", source["text"])],
        "sources": sources,
    }
    report.append(entry)
    pages = ", ".join(str(page) for page in sorted({source["page"] for source in sources}))
    print(f"{number} | PDF {pages} 页 | {entry['condition'] or '待复核'}")

expected = {f"8.2.{i}" for i in range(1, 13)}
team_subitems = next((len(item["subitems"]) for item in report if item["number"] == "8.2.3"), 0)
print(f"\n一级条款：{len(expected & clauses.keys())}/12")
print(f"串通投标子项：{team_subitems}/6")
print("缺少编号：", sorted(expected - clauses.keys()))
print("多出编号：", sorted((clauses.keys() | summaries.keys()) - expected))

output = DATA / f"{DOCUMENT}_invalid_bid_report.json"
output.write_text(
    json.dumps({"document": f"{DOCUMENT}.pdf", "section": "8.2", "items": report}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"清单已保存到 {output.name}")